In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss
from lightgbm import LGBMClassifier

In [2]:
target_cols = [f'target_{i}' for i in range(7)]

In [3]:
BASE_DIR = "/kaggle/input/h2o-predict-the-llm"

In [4]:
train = pd.read_csv(BASE_DIR + '/train_oof_mistral.csv')

train2 = pd.read_csv(BASE_DIR + '/oof_encoder.csv')
train2 = train2.rename(columns={k: 'sub2_' + k for k in target_cols })

train3 = pd.read_csv(BASE_DIR + '/train_oof_zephyr.csv')
train3 = train3.rename(columns={k: 'sub3_' + k for k in target_cols })

In [5]:
sub2_target_cols = ['sub2_'+ t for t in target_cols]
sub3_target_cols = ['sub3_'+ t for t in target_cols]

In [6]:
test = pd.read_csv("/kaggle/input/h2oai-predict-the-llm/test.csv")
test[target_cols] = pd.read_csv(BASE_DIR + '/sub_LoRa_mistral_5folds.csv')[target_cols]
test["Response"] = test["Response"].fillna("NA")

test2 = pd.read_csv('/kaggle/input/h2oai-predict-the-llm/test.csv')
test2["Response"] = test2["Response"].fillna("NA")
test2[sub2_target_cols] = pd.read_csv(BASE_DIR + '/sub_encoder.csv')[target_cols]


test3 = pd.read_csv("/kaggle/input/h2oai-predict-the-llm/test.csv")
test3["Response"] = test3["Response"].fillna("NA")
test3[sub3_target_cols] = pd.read_csv(BASE_DIR + '/sub_LoRa_zephyr_5folds.csv')[target_cols]


In [7]:
train['Response'] = train['Response'].fillna("NA")

In [8]:
def count_string(x):
    return len(x.split())

train["word_count_q"]= train.Question.apply(count_string)
train["word_count_r"]= train.Response.apply(count_string)

test["word_count_q"]= test.Question.apply(count_string)
test["word_count_r"]= test.Response.apply(count_string)

In [9]:
df = train.copy()
rows = []
added = [f'target_{i}_r{j}' for j in range(6) for i in range(7)]
for i, row in df.iterrows():
    question = row.Question
    response = row.Response
    context_df = train[(train.Question==question)&(train.Response!=response)].reset_index()
    other_responses_preds = []
    for j, rrow in context_df.iterrows():
        df.loc[i,  [f'target_{k}_r{j}' for k in range(7)]] = rrow[target_cols].values
        df.loc[i, f'wc_r{j}'] = rrow["word_count_r"]

test_ = test.copy()
rows = []
added = [f'target_{i}_r{j}' for j in range(6) for i in range(7)]
for i, row in test_.iterrows():
    question = row.Question
    response = row.Response
    context_df = test[(test.Question==question)&(test.Response!=response)].reset_index()
    other_responses_preds = []
    for j, rrow in context_df.iterrows():
        test_.loc[i,  [f'target_{k}_r{j}' for k in range(7)]] = rrow[target_cols].values
        test_.loc[i, f'wc_r{j}'] = rrow["word_count_r"]

for i in range(7):
    df[f'f1_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] ].std(axis=1)
    df[f'f2_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    df[f'f3_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] ].mean(axis=1)
    df[f'f4_{i}'] = 1 - df[[f'target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    df[f'f5_{i}'] = df[[f'target_{i}_r{j}' for j in range(6)]].mean(axis=1)

for i in range(7):
    test_[f'f1_{i}'] =test_[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] ].std(axis=1)
    test_[f'f2_{i}'] =test_[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    test_[f'f3_{i}'] =test_[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] ].mean(axis=1)
    test_[f'f4_{i}'] = 1 - test_[[f'target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    test_[f'f5_{i}'] = test_[[f'target_{i}_r{j}' for j in range(6)] ].mean(axis=1)
        
        

In [10]:
df2 = train2.copy()
rows = []
added = [f'sub2_target_{i}_r{j}' for j in range(6) for i in range(7)]
for i, row in df2.iterrows():
    question = row.Question
    response = row.Response
    context_df = train2[(train2.Question==question)&(train2.Response!=response)].reset_index()
    other_responses_preds = []
    for j, rrow in context_df.iterrows():
        df2.loc[i,  [f'sub2_target_{k}_r{j}' for k in range(7)]] = rrow[sub2_target_cols].values

test_2 = test2.copy()
rows = []
added = [f'sub2_target_{i}_r{j}' for j in range(6) for i in range(7)]
for i, row in test_2.iterrows():
    question = row.Question
    response = row.Response
    context_df = test2[(test2.Question==question)&(test2.Response!=response)].reset_index()
    other_responses_preds = []
    for j, rrow in context_df.iterrows():
        test_2.loc[i,  [f'sub2_target_{k}_r{j}' for k in range(7)]] = rrow[sub2_target_cols].values
        
for i in range(7):
    df2[f'sub2_f1_{i}'] = df2[[f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] ].std(axis=1)
    df2[f'sub2_f2_{i}'] = df2[[f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    df2[f'sub2_f3_{i}'] = df2[[f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)
    df2[f'sub2_f4_{i}'] = 1 - df2[[f'sub2_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    df2[f'sub2_f5_{i}'] = df2[[f'sub2_target_{i}_r{j}' for j in range(6)]].mean(axis=1)

for i in range(7):
    test_2[f'sub2_f1_{i}'] =test_2[[f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] ].std(axis=1)
    test_2[f'sub2_f2_{i}'] =test_2[[f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    test_2[f'sub2_f3_{i}'] =test_2[[f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)
    test_2[f'sub2_f4_{i}'] = 1 - test_2[[f'sub2_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    test_2[f'sub2_f5_{i}'] = test_2[[f'sub2_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)

In [11]:
df3 = train3.copy()
rows = []
added = [f'sub3_target_{i}_r{j}' for j in range(6) for i in range(7)]
for i, row in df3.iterrows():
    question = row.Question
    response = row.Response
    context_df = train3[(train3.Question==question)&(train3.Response!=response)].reset_index()
    other_responses_preds = []
    for j, rrow in context_df.iterrows():
        df3.loc[i,  [f'sub3_target_{k}_r{j}' for k in range(7)]] = rrow[sub3_target_cols].values

test_3 = test3.copy()
rows = []
added = [f'sub3_target_{i}_r{j}' for j in range(6) for i in range(7)]
for i, row in test_3.iterrows():
    question = row.Question
    response = row.Response
    context_df = test3[(test3.Question==question)&(test3.Response!=response)].reset_index()
    other_responses_preds = []
    for j, rrow in context_df.iterrows():
        test_3.loc[i,  [f'sub3_target_{k}_r{j}' for k in range(7)]] = rrow[sub3_target_cols].values
        
for i in range(7):
    df3[f'sub3_f1_{i}'] = df3[[f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].std(axis=1)
    df3[f'sub3_f2_{i}'] = df3[[f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    df3[f'sub3_f3_{i}'] = df3[[f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)
    df3[f'sub3_f4_{i}'] = 1 - df3[[f'sub3_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    df3[f'sub3_f5_{i}'] = df3[[f'sub3_target_{i}_r{j}' for j in range(6)]].mean(axis=1)

for i in range(7):
    test_3[f'sub3_f1_{i}'] =test_3[[f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].std(axis=1)
    test_3[f'sub3_f2_{i}'] =test_3[[f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    test_3[f'sub3_f3_{i}'] =test_3[[f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)
    test_3[f'sub3_f4_{i}'] = 1 - test_3[[f'sub3_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    test_3[f'sub3_f5_{i}'] = test_3[[f'sub3_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)

In [12]:
df = pd.concat([df, df2, df3], axis=1)
test_ = pd.concat([test_, test_2, test_3], axis=1)

In [13]:
for i in range(7):
    df[f'subs_f1_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] + 
                           [f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].std(axis=1)
    df[f'subs_f2_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] +
                          [f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)]  ].sum(axis=1)
    
    df[f'subs_f3_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] + 
                           [f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)


for i in range(7):
    test_[f'subs_f1_{i}'] =test_[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] + 
                           [f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)]].std(axis=1)
    
    test_[f'subs_f2_{i}'] =test_[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] +
                          [f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].sum(axis=1)
    
    test_[f'subs_f3_{i}'] =test_[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] + 
                           [f'sub3_target_{i}'] + [f'sub3_target_{i}_r{j}' for j in range(6)] ].mean(axis=1)

/tmp/ipykernel_20/3814397712.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'subs_f1_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] +
/tmp/ipykernel_20/3814397712.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'subs_f2_{i}'] = df[[f'target_{i}'] + [f'target_{i}_r{j}' for j in range(6)] + [f'sub2_target_{i}'] + [f'sub2_target_{i}_r{j}' for j in range(6)] +
/tmp/ipykernel_20/3814397712.py:7: PerformanceWarning: DataFrame is 

In [14]:
FEATURES = df.columns.difference(["Question", "Response", "target", "context", "fold", "input", "label"] + ['sub2_target_0_r6', 'sub2_target_1_r6', 'sub2_target_2_r6', 'sub2_target_3_r6', 'sub2_target_4_r6', 'sub2_target_5_r6', 'sub2_target_6_r6', 'sub3_target_0_r6', 'sub3_target_1_r6', 'sub3_target_2_r6', 'sub3_target_3_r6', 'sub3_target_4_r6', 'sub3_target_5_r6', 'sub3_target_6_r6']).tolist()

In [15]:
len(FEATURES)

281

In [16]:
y = df['label'].iloc[:, 0].copy()

In [17]:
del df['label']

In [18]:
df['label'] = y

/tmp/ipykernel_20/432619920.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['label'] = y


In [19]:
from sklearn.metrics import log_loss
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
final_preds = []
scores = []
for fold in range(5):
    train_ = df.query(f"fold != {fold}").reset_index(drop=True)
    valid = df.query(f"fold == {fold}").reset_index(drop=True)
    
    model = LGBMClassifier(n_estimators=3500, early_stopping_rounds=120, verbose=-1, max_depth=3, subsample=0.7, feature_fraction=0.65)
    model2 = XGBClassifier(n_estimators=3500, early_stopping_rounds=120, max_depth=5, subsample=0.75, colsample_bytree=0.5)
    model.fit(train_[FEATURES], train_['label'], eval_set=[(valid[FEATURES], valid['label'])])
    model2.fit(train_[FEATURES], train_['label'], eval_set=[(valid[FEATURES], valid['label'])],verbose=300)
    
    preds = model.predict_proba(valid[FEATURES])*0.75 + model2.predict_proba(valid[FEATURES])*0.25
    metric = log_loss(valid['label'], preds)
    print(f"Fold {fold+1}", metric)
    pred = model.predict_proba(test_[FEATURES])*0.75 + model2.predict_proba(test_[FEATURES])*0.25
    final_preds.append(pred)
    scores.append(metric)
print(f"scores: {scores}")
print(f"mean: {np.mean(scores)}")

[LightGBM] [Warning] early_stopping_round is set=120, early_stopping_rounds=120 will be ignored. Current value: early_stopping_round=120
[LightGBM] [Warning] feature_fraction is set=0.65, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.65
[1]	valid_0's multi_logloss: 1.65595
[2]	valid_0's multi_logloss: 1.47081
[3]	valid_0's multi_logloss: 1.32812
[4]	valid_0's multi_logloss: 1.22186
[5]	valid_0's multi_logloss: 1.1355
[6]	valid_0's multi_logloss: 1.06402
[7]	valid_0's multi_logloss: 1.00493
[8]	valid_0's multi_logloss: 0.952747
[9]	valid_0's multi_logloss: 0.91166
[10]	valid_0's multi_logloss: 0.873917
[11]	valid_0's multi_logloss: 0.844105
[12]	valid_0's multi_logloss: 0.815608
[13]	valid_0's multi_logloss: 0.791578
[14]	valid_0's multi_logloss: 0.77182
[15]	valid_0's multi_logloss: 0.753537
[16]	valid_0's multi_logloss: 0.737368
[17]	valid_0's multi_logloss: 0.723763
[18]	valid_0's multi_logloss: 0.711719
[19]	valid_0's multi_logloss: 0.702299
[20]	valid_0's 

In [20]:
sub = test.copy()
sub[target_cols] = np.mean(final_preds, 0)

In [21]:
sub[['id'] + target_cols]

,id,target_0,target_1,target_2,target_3,target_4,target_5,target_6
0,0,0.132752,0.128430,0.147188,0.011914,0.251611,0.191756,0.136349
1,1,0.000157,0.000493,0.000143,0.000065,0.000095,0.998873,0.000175
2,2,0.000162,0.000128,0.000114,0.000070,0.000097,0.999264,0.000165
3,3,0.758075,0.006697,0.051061,0.003912,0.058829,0.000970,0.120458
4,4,0.000424,0.043825,0.000415,0.954024,0.000245,0.000125,0.000942
...,...,...,...,...,...,...,...,...
996,996,0.000160,0.000136,0.000157,0.000079,0.000130,0.999177,0.000161
997,997,0.004166,0.960031,0.008355,0.009636,0.001149,0.001699,0.014963
998,998,0.251994,0.006134,0.364520,0.001380,0.356886,0.001037,0.018048
999,999,0.556674,0.001848,0.207728,0.003754,0.221990,0.000965,0.007041


In [22]:
sub[['id'] + target_cols].to_csv('submission.csv', index=False)